In [1]:
from tkinter import *

# CONSTANTS

In [2]:
PINK = "#e2979c"
RED = "#e7305b"
GREEN = "#9bdeac"
YELLOW = "#f7f5dd"
FONT_NAME = "Courier"
WORK_MIN = 25
SHORT_BREAK_MIN = 5
LONG_BREAK_MIN = 20

# 215. UI SETUP

## Window

In [3]:
# Window
window = Tk()
window.title("Pomodoro")
window.config(padx=100, pady=50, bg=YELLOW)

## Label1: time_label 显示时间

In [4]:
## Label1
timer_label = Label(text="Timer", font=(FONT_NAME, 50), fg=GREEN, bg=YELLOW)
timer_label.grid(column=1, row=0)

## Canvas

In [5]:
## Canvas
canvas = Canvas(width=200, height=224, bg=YELLOW, highlightthickness=0)

调用 `Canvas` Class 的构造方法 `__init__` ，生成一个 `Canvas` Class 的实例，并让 `canvas` 指向它。

- `width=200, height=244`: 设置该 `Canvas` Class 实例 `canvas` 的宽度和高度。
- `bg=YELLOW`: 设置 `canvas` 这块画布本身的背景颜色为黄色，也就是说，如果画布上什么都没有，它就是一整块黄色区域。
- `highlightthickness`: 如果这个不设成0，那么`canvas`周围就会有一圈白色。

In [6]:
tomato_img = PhotoImage(file="tomato.png")

- `PhotoImage(file="tomato.png")`：
  
    这样的写法，就是在调用 `tkinter` Package 下的 `PhotoImage` Class 的构造方法 `__init__`。

    构造方法的形式参数`file`：把 *同级目录* 下的 `"tomato.png"` 作为参数传给`file`。
  
    由此生成一个 `PhotoImage` Class 的实例。
  
之所以这么做，是因为 `tkinter`Package 中要操作图片，就是通过 `PhotoImage` Class 的实例来操作的，而不是直接操作一个JPEG或者PNG图片的。

- `tomato_img = PhotoImage(file="tomato.png")`

    让`tomato_img`指向这个`PhotoImage`类的实例。

In [7]:
canvas.create_image(100, 112, image=tomato_img)

#### 解释1：侧重背景颜色、像素方面的理解

在这块黄色画布`canvas`的坐标 `(100, 112)` 处放上 `PhotoImage` Class 对象 `tomato_img`，即为在该处放上番茄图片 `"tomato.png"`。

- 如果 `"tomato.png"` 的背景是透明的，那么透明部分会露出下面 `canvas` 的黄色背景，如果图片背景不是透明的，那图片自己的背景颜色就会盖住 `canvas`。
    - 番茄图片 `"tomato.png"` 本身有自己的像素颜色，所以它会覆盖 `canvas` 对应区域。

#### 解释2：侧重OOP方面的理解

- `create_image()` 是 Canvas 对象的方法，用来在这块 canvas 上创建一个图片 item。
- 它内部会调用 Canvas 的内部方法 `_create('image', ...)`，真正把图片 item 创建到画布上。
- 返回值是这个图片 item 在 canvas 里的编号 ID，不过这里我们没有保存它。

In [ ]:
timer_text = canvas.create_text(100, 130, text="00:00", fill="white", font=(FONT_NAME, 35, "bold"))

### 到了这里可以注意一个共性：     （注：这个cell的内容都属于深挖和深入理解，消耗精神能量大，精力不够不要硬碰）
- 在canvas上想放置图片，就用`create_image()`
- 在canvas上想放置文本，就用`create_text()`

`create_image()` 和 `create_text()` 不是类的构造方法，它们只是 `Canvas` 对象上的普通方法。它们的作用是：告诉这个已经存在的 `canvas`，在画布上创建一个 “图片 item” 或者 “文字 item” 。

#### 详细来说：

```python
canvas.create_image(100, 112, image=tomato_img)
```
不是在创建一个新的 Python Image 类实例，而是在 canvas 这块画布里面创建一个 Canvas item，类型是 "image"。

```python
canvas.create_text(100, 130, text="00:00", ...)
```
是在 canvas 里面创建一个 Canvas item，类型是 "text"。


#### 而如果在Python里面去看 `create_image` 和 `create_text` 的 方法定义：

```python
def create_image(self, *args, **kw):
    """Create image item with coordinates x1,y1."""
    return self._create('image', args, kw)

def create_text(self, *args, **kw):
    """Create text with coordinates x1,y1."""
    return self._create('text', args, kw)
```

可以看出来，这两个方法都在调用一个更底层的方法 `_create` 。

因此，事实上，`create_image()` 和 `create_text()` 是给用户用的封装方法，真正统一处理创建工作的，是内部辅助方法 `_create()`。

`_create()` 里的下划线 `_` 表示：这是一个内部使用的方法，不是给初学者直接调用的公共接口。
- 即，平时不应该直接通过 `Canvas` Class 对象来直接调用这个方法，
    - 如 `canvas._create(...)` 就是不被推荐的。
- 而应该通过封装方法来调用：
    - 如课程中示例的 `canvas.create_image(...)`  `canvas.create_text(...)`

#### 【`_create(...)`到底是什么？】  和   【item的编号ID】

`_create()` 是何方神圣，简单说就是：它负责把 Python 里的参数整理好，然后交给更加底层的 Tk/Tcl 去真正创建 Canvas item。Tkinter 其实是 Python 对 Tcl/Tk 图形库的一层包装，所以很多真正的 GUI 操作最后都会变成对底层 Tk 的调用。

可以简单理解成：

```
你写 Python 代码
↓
canvas.create_text(...)
↓
Tkinter 内部调用 canvas._create(...)
↓
_create 把参数翻译给底层 Tk
↓
Tk 在窗口上真正生成文字
↓
返回一个 item id 给 Python
```

这个**返回值**也很重要。

```python
timer_text = canvas.create_text(100, 130, text="00:00", fill="white", font=(FONT_NAME, 35, "bold"))
```

这里的 `timer_text` 不是文字对象本身，也不是 `"00:00"` 这个字符串，而是 Canvas 里这个文字 item 的 **编号 ID**。正是因为有了这个ID，后面（见216节）才能用：

```python
canvas.itemconfig(timer_text, text=count)
```

意思就是：“找到编号为 timer_text 的那个 Canvas item，把它显示的文字改掉。”

In [ ]:
canvas.grid(column=1, row=1)

## Buttons: start_button  reset_button  计时开始按钮  时间重置按钮

In [8]:
## Button1
start_button = Button(text="Start", highlightthickness=0)
start_button.grid(column=0, row=2)

## Button2
reset_button = Button(text="Reset", highlightthickness=0)
reset_button.grid(column=2, row=2)

## Label2: Check marks  完成了多少个番茄钟

In [9]:
# Label2
check_marks = Label(text="✔", fg=GREEN, bg=YELLOW)
check_marks.grid(column=1, row=3)

## 开启窗口进程，监控用户行为

In [ ]:
window.mainloop()